# Conditional analysis from SCWF coefficient stores

This notebook consumes the **coefficient-only** output of the revised pipeline.

Use the saved long-form `CoefficientStore` rows, not interval-averaged `Spectra`, `Sfuncs`, or `WaveletMoments`.

Second-order conditional spectra are estimated from
\[
\widehat{P} = \left\langle \frac{|a|^2}{A_j} \right\rangle,
\qquad
A_j = \texttt{response\_energy\_integral},
\]
and higher-order scale-normalized moments are estimated from
\[
\widehat{M}_q = \left\langle \frac{|a|^q}{\tau_{\mathrm{eq}}^{q/2}} \right\rangle.
\]

Directional buckets are reconstructed later from the saved local angles `thetas` and `phis`.


In [5]:
pwd

'C:\\Users\\nokni\\work\\MHDTurbPy\\functions'

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
os.chdir('C:\\Users\\nokni\\work\\MHDTurbPy\\functions\\scwf_cleaned_pkg_rev12\\')
from cond_analysis_scwf import load_and_merge_coefficient_stores, reduce_conditional_rows

In [13]:
# Edit this glob to point to your saved first-pass interval pickles
PATHS = r'C:\Users\nokni\work\WIND_3D\data\3_sec\*\\anisotropy_scwf\\general_SF_scwf_mw8_extra_conditions_theta_0_phi_0.pkl'

# Load one merged long-form dataframe across all intervals/scales
df = load_and_merge_coefficient_stores(PATHS)
df.shape

KeyError: 'C:\\Users\\nokni\\work\\WIND_3D\\data\\3_sec\\1999-08-26_23-30-00_1999-08-29_01-08-00_sc_0\\anisotropy_scwf\\general_SF_scwf_mw8_extra_conditions_theta_0_phi_0.pkl does not contain CoefficientStore/CompactCoefficients.'

In [ ]:
df.head()

## Example 1: conditional B-based spectra for wave-vector anisotropy

Use the same scalar B observable across different directional buckets.

Examples:
- `W_B_mag` for the magnetic trace coefficient magnitude,
- `B_perp` for the perpendicular magnetic coefficient magnitude.

The compressibility conditions are usually the samplewise columns:
- `compress_simple_parallel`,
- `compress_simple_absB`.


In [ ]:
scale_edges = np.geomspace(1.0, 1.0e4, 41)

psd_ell_perp = reduce_conditional_rows(
    df,
    bucket='ell_perp',
    value_keys=['W_B_mag', 'B_perp'],
    cond_var='compress_simple_absB',
    qorder=[2.0],
    normalization='psd',
    scale_bin_edges_di=scale_edges,
    constraints={'coi_mask': (0.5, None), 'is_effective_level': (0.5, None)},
    min_count=25,
    nquant=10,
)
psd_ell_perp.head()


## Example 2: scale-normalized higher-order moments

For B-only analysis, use `W_B_mag`, `B_perp`, and optionally `B_par` if you saved it.

These are the quantities that can be interpreted as **structure-function surrogates** with the correct physical units `X^q`.


In [ ]:
mom_ell_par = reduce_conditional_rows(
    df,
    bucket='ell_par',
    value_keys=['W_B_mag', 'B_perp'],
    cond_var='compress_simple_parallel',
    qorder=[1.0, 2.0, 3.0, 4.0],
    normalization='scale_normalized',
    scale_bin_edges_di=scale_edges,
    constraints={'coi_mask': (0.5, None), 'is_effective_level': (0.5, None)},
    min_count=25,
    nquant=10,
)
mom_ell_par.head()


## Notes

- For **wave-vector anisotropy**, compare the same scalar B observable across `ell_perp`, `Ell_perp`, and `ell_par`.
- For **component scaling**, compare `B_perp` and, if saved, `B_par` within a fixed bucket.
- `compress_simple_parallel = |B_parallel|^2 / |B_trace|^2` is a samplewise magnetic compressibility surrogate.
- `compress_simple_absB = |\delta |B||^2 / |B_trace|^2` is the scalar-|B| compressibility surrogate.
- Do not use already averaged outputs as input for a new condition. Use the raw row store.
